# 📊 Trial Size Impact: Comprehensive Long-Term Customer Value Analysis

## 分析目的
トライアル時のサイズが顧客の長期的なロイヤリティに与える影響を多角的に評価する

## 仮説
「大きいサイズでトライアルした顧客は、短期リピート率は低いが、実際にはより高いロイヤリティを持っている」

## 分析フレームワーク
1. **Phase 1**: 6ヶ月リピート率（期間延長）
2. **Phase 2**: Time-to-Repeat（リピートまでの日数）
3. **Phase 3**: 定着率（Retention）
4. **Phase 4**: Share of Wallet（SOW）
5. **Phase 5**: Customer Lifetime Value（CLV）
6. **Phase 6**: Size Migration（サイズ遷移）
7. **Phase 7**: Cohort Analysis（月次追跡）
8. **Phase 8**: Integrated Summary（統合評価）

## 1. Setup and Configuration

In [1]:
# =============================================================================
# Import Libraries and Setup
# =============================================================================

from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load environment variables
load_dotenv(dotenv_path='../../.env')

# Validate credentials
required_vars = ['DATABRICKS_HOST', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")
print(f"  Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")

✓ Environment configured
  Analysis Date: 2026-01-30


In [2]:
# =============================================================================
# Initiative Configurations with 6-Month Extended Repeat Period
# =============================================================================

# Current date for calculating observation period
CURRENT_DATE = datetime(2026, 1, 30)
REQUIRED_OBSERVATION_MONTHS = 6

def calculate_observation_months(pre_end_date):
    """Calculate months of observation available from pre_end to current date"""
    pre_end = datetime.strptime(pre_end_date, '%Y-%m-%d')
    delta = CURRENT_DATE - pre_end
    return delta.days / 30.44  # Average days per month

def calculate_6month_repeat_end(pre_end_date):
    """Calculate 6-month repeat end date from pre_end"""
    pre_end = datetime.strptime(pre_end_date, '%Y-%m-%d')
    repeat_end = pre_end + timedelta(days=180)  # 6 months
    return repeat_end.strftime('%Y-%m-%d')

# Initiative configurations
initiatives = [
    # ============= ARIEL GEL BALL =============
    {
        'initiative_name': 'Srixon',
        'brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-04-13',
        'pre_end': '2024-05-13',
    },
    {
        'initiative_name': 'Srixon Boost',
        'brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-09-07',
        'pre_end': '2024-10-07',
    },
    {
        'initiative_name': 'Yoda',
        'brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-02-17',
        'pre_end': '2025-03-19',
    },
    {
        'initiative_name': 'Anakin (All)',
        'brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-11-01',
        'pre_end': '2025-12-01',
    },
    # ============= BOLD GEL BALL =============
    {
        'initiative_name': 'Rapunzel (All)',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
    },
    {
        'initiative_name': 'Cinderella (All)',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-10-01',
        'pre_end': '2024-10-31',
    },
    {
        'initiative_name': 'Moana',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-02-01',
        'pre_end': '2024-03-02',
    },
    {
        'initiative_name': 'Snowwhite',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-04-12',
        'pre_end': '2025-05-12',
    },
    # ============= LIQUID DETERGENTS =============
    {
        'initiative_name': 'Ariel Gel (Liquid)',
        'brand': 'Ariel Gel',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙ',
        'pre_start': '2025-03-17',
        'pre_end': '2025-04-17',
    },
    {
        'initiative_name': 'Bold Gel (Liquid)',
        'brand': 'Bold Gel',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
    },
    # ============= COMPETITORS =============
    {
        'initiative_name': 'Attack Antibacterial EX',
        'brand': 'Attack Antibacterial EX',
        'sub_brand': 'ｱﾀｯｸ抗菌EX',
        'pre_start': '2025-07-05',
        'pre_end': '2025-08-04',
    },
    {
        'initiative_name': 'Nanox One',
        'brand': 'Nanox One',
        'sub_brand': 'ﾅﾉｯｸｽﾜﾝ',
        'pre_start': '2025-09-25',
        'pre_end': '2025-10-24',
    },
]

# Add calculated fields
for init in initiatives:
    init['observation_months'] = calculate_observation_months(init['pre_end'])
    init['has_6month_data'] = init['observation_months'] >= REQUIRED_OBSERVATION_MONTHS
    init['repeat_end_6month'] = calculate_6month_repeat_end(init['pre_end'])
    # Cap repeat_end at current date if needed
    repeat_end_dt = datetime.strptime(init['repeat_end_6month'], '%Y-%m-%d')
    if repeat_end_dt > CURRENT_DATE:
        init['repeat_end_actual'] = CURRENT_DATE.strftime('%Y-%m-%d')
    else:
        init['repeat_end_actual'] = init['repeat_end_6month']

# Customer filters
customer_codes = ['cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009', 
                  'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013']
customer_filter_sql = "', '".join(customer_codes)

# Display configuration
print("="*80)
print("INITIATIVE CONFIGURATIONS WITH OBSERVATION STATUS")
print("="*80)
print(f"\nCurrent Date: {CURRENT_DATE.strftime('%Y-%m-%d')}")
print(f"Required Observation Period: {REQUIRED_OBSERVATION_MONTHS} months")
print("\n" + "-"*80)
print(f"{'Initiative':<30} | {'Pre End':<12} | {'Obs Months':<10} | {'Status':<20}")
print("-"*80)

for init in initiatives:
    status = "✅ Full 6-month data" if init['has_6month_data'] else f"⚠️ {init['observation_months']:.1f} months only"
    print(f"{init['initiative_name']:<30} | {init['pre_end']:<12} | {init['observation_months']:<10.1f} | {status}")

print("-"*80)
full_data_count = sum(1 for i in initiatives if i['has_6month_data'])
print(f"\n✅ Full 6-month data available: {full_data_count}/{len(initiatives)} initiatives")
print(f"⚠️ Partial data (will be marked): {len(initiatives) - full_data_count}/{len(initiatives)} initiatives")

INITIATIVE CONFIGURATIONS WITH OBSERVATION STATUS

Current Date: 2026-01-30
Required Observation Period: 6 months

--------------------------------------------------------------------------------
Initiative                     | Pre End      | Obs Months | Status              
--------------------------------------------------------------------------------
Srixon                         | 2024-05-13   | 20.6       | ✅ Full 6-month data
Srixon Boost                   | 2024-10-07   | 15.8       | ✅ Full 6-month data
Yoda                           | 2025-03-19   | 10.4       | ✅ Full 6-month data
Anakin (All)                   | 2025-12-01   | 2.0        | ⚠️ 2.0 months only
Rapunzel (All)                 | 2025-10-31   | 3.0        | ⚠️ 3.0 months only
Cinderella (All)               | 2024-10-31   | 15.0       | ✅ Full 6-month data
Moana                          | 2024-03-02   | 23.0       | ✅ Full 6-month data
Snowwhite                      | 2025-05-12   | 8.6        | ✅ Full 6-month 

In [3]:
# =============================================================================
# Database Connection Helper
# =============================================================================

def get_db_connection():
    """Create Databricks connection"""
    return sql.connect(
        server_hostname=os.getenv('DATABRICKS_HOST'),
        http_path=os.getenv('DATABRICKS_HTTP_PATH'),
        access_token=os.getenv('DATABRICKS_TOKEN')
    )

print("✓ Database connection helper defined")

✓ Database connection helper defined


## 2. Phase 1: Extended 6-Month Repeat Rate Analysis

### 目的
- リピート観測期間を6ヶ月に延長
- 個別サイズごとのリピート率を算出
- 観測期間不足の商品には⚠️マークを付与

In [ ]:
# =============================================================================
# Phase 1: Extract Trial and Repeat Data with 6-Month Window
# =============================================================================

print("="*80)
print("PHASE 1: EXTRACTING 6-MONTH REPEAT DATA")
print("="*80)

# Build comprehensive query for all initiatives
def build_trial_repeat_query(initiatives):
    """Build SQL query to extract trial shoppers with size and repeat info"""
    
    case_clauses = []
    date_filters = []
    
    for init in initiatives:
        case_clause = f"""
            WHEN jp_sub_brand_alter_lang_name = '{init['sub_brand']}' 
                 AND sales_period_group_end_date_part BETWEEN '{init['pre_start']}' AND '{init['pre_end']}'
            THEN '{init['initiative_name']}'
        """
        case_clauses.append(case_clause)
        
        date_filter = f"""
            (jp_sub_brand_alter_lang_name = '{init['sub_brand']}' 
             AND sales_period_group_end_date_part BETWEEN '{init['pre_start']}' AND '{init['repeat_end_actual']}')
        """
        date_filters.append(date_filter)
    
    query = f"""
    WITH all_transactions AS (
        SELECT
            idpos.shopper_key,
            sales_period_group_end_date_part AS txn_date,
            jp_sub_brand_alter_lang_name AS sub_brand,
            jp_segment_4_name AS pack_size,
            pos_sales_amt,
            CASE 
                {' '.join(case_clauses)}
                ELSE NULL
            END AS trial_initiative
        FROM
            cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
            LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
            LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
        WHERE
            jp_category_name = 'Laundry'
            AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
            AND shopper.member_ind = 'Y'
            AND ({' OR '.join(date_filters)})
    ),
    trial_shoppers AS (
        SELECT 
            shopper_key,
            trial_initiative AS initiative_name,
            pack_size AS trial_size,
            MIN(txn_date) AS first_trial_date,
            SUM(net_spend_amt) AS trial_spend
        FROM all_transactions
        WHERE trial_initiative IS NOT NULL
        GROUP BY shopper_key, trial_initiative, pack_size
    ),
    repeat_info AS (
        SELECT
            t.shopper_key,
            t.initiative_name,
            t.trial_size,
            t.first_trial_date,
            t.trial_spend,
            COUNT(DISTINCT r.txn_date) AS repeat_purchase_count,
            MIN(r.txn_date) AS first_repeat_date,
            MAX(r.txn_date) AS last_repeat_date,
            SUM(r.net_spend_amt) AS repeat_spend
        FROM trial_shoppers t
        LEFT JOIN all_transactions r 
            ON t.shopper_key = r.shopper_key
            AND r.sub_brand = (
                SELECT sub_brand FROM all_transactions 
                WHERE shopper_key = t.shopper_key AND trial_initiative = t.initiative_name
                LIMIT 1
            )
            AND r.txn_date > t.first_trial_date
        GROUP BY t.shopper_key, t.initiative_name, t.trial_size, t.first_trial_date, t.trial_spend
    )
    SELECT * FROM repeat_info
    """
    return query

print("\n📊 Building and executing data extraction query...")
print("   This query extracts trial shoppers with size and 6-month repeat info")

PHASE 1: EXTRACTING 6-MONTH REPEAT DATA

📊 Building and executing data extraction query...
   This query extracts trial shoppers with size and 6-month repeat info


In [7]:
# =============================================================================
# Execute Simplified Extraction - Trial Size with Repeat Status
# =============================================================================

print("="*80)
print("EXTRACTING TRIAL SIZE AND REPEAT DATA")
print("="*80)

# Simpler, more efficient query approach - one query per initiative group
def extract_trial_repeat_data():
    """Extract trial and repeat data for all initiatives"""
    
    all_results = []
    
    # Group initiatives by sub_brand for efficient querying
    sub_brand_groups = {}
    for init in initiatives:
        sb = init['sub_brand']
        if sb not in sub_brand_groups:
            sub_brand_groups[sb] = []
        sub_brand_groups[sb].append(init)
    
    for sub_brand, inits in sub_brand_groups.items():
        print(f"\n📊 Extracting data for {sub_brand}...")
        
        # Build date conditions for this sub-brand
        trial_conditions = []
        repeat_conditions = []
        
        for init in inits:
            trial_conditions.append(
                f"(sales_period_group_end_date_part BETWEEN '{init['pre_start']}' AND '{init['pre_end']}')"
            )
            repeat_conditions.append(
                f"(sales_period_group_end_date_part BETWEEN '{init['pre_end']}' AND '{init['repeat_end_actual']}')"
            )
        
        query = f"""
        WITH trial_txns AS (
            SELECT
                idpos.shopper_key,
                sales_period_group_end_date_part AS txn_date,
                jp_segment_4_name AS pack_size,
                pos_sales_amt,
                CASE 
                    {' '.join([f"WHEN sales_period_group_end_date_part BETWEEN '{i['pre_start']}' AND '{i['pre_end']}' THEN '{i['initiative_name']}'" for i in inits])}
                END AS initiative_name
            FROM
                cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
                LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
                LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
            WHERE
                jp_category_name = 'Laundry'
                AND jp_sub_brand_alter_lang_name = '{sub_brand}'
                AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
                AND shopper.member_ind = 'Y'
                AND ({' OR '.join(trial_conditions)})
        ),
        trial_shoppers AS (
            SELECT 
                shopper_key,
                initiative_name,
                pack_size AS trial_size,
                MIN(txn_date) AS first_trial_date
            FROM trial_txns
            WHERE initiative_name IS NOT NULL
            GROUP BY shopper_key, initiative_name, pack_size
        ),
        repeat_txns AS (
            SELECT
                idpos.shopper_key,
                sales_period_group_end_date_part AS txn_date,
                jp_segment_4_name AS repeat_size,
                pos_sales_amt
            FROM
                cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
                LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
                LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
            WHERE
                jp_category_name = 'Laundry'
                AND jp_sub_brand_alter_lang_name = '{sub_brand}'
                AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
                AND shopper.member_ind = 'Y'
                AND ({' OR '.join(repeat_conditions)})
        ),
        final AS (
            SELECT
                t.shopper_key,
                t.initiative_name,
                t.trial_size,
                t.first_trial_date,
                MAX(CASE WHEN r.txn_date IS NOT NULL THEN 1 ELSE 0 END) AS has_repeat,
                MIN(r.txn_date) AS first_repeat_date,
                COUNT(DISTINCT r.txn_date) AS repeat_count,
                SUM(r.pos_sales_amt) AS repeat_spend
            FROM trial_shoppers t
            LEFT JOIN repeat_txns r 
                ON t.shopper_key = r.shopper_key
                AND r.txn_date > t.first_trial_date
            GROUP BY t.shopper_key, t.initiative_name, t.trial_size, t.first_trial_date
        )
        SELECT * FROM final
        """
        
        try:
            with get_db_connection() as conn:
                with conn.cursor() as cursor:
                    cursor.execute(query)
                    result = cursor.fetchall()
                    columns = [desc[0] for desc in cursor.description]
                    df = pd.DataFrame(result, columns=columns)
                    all_results.append(df)
                    print(f"  ✓ Extracted {len(df):,} records")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# Execute extraction
import time
start_time = time.time()
trial_repeat_df = extract_trial_repeat_data()
extraction_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"✓ Data extraction completed in {extraction_time:.1f} seconds")
print(f"  Total records: {len(trial_repeat_df):,}")
print(f"  Unique shoppers: {trial_repeat_df['shopper_key'].nunique():,}")
print(f"  Initiatives: {trial_repeat_df['initiative_name'].nunique()}")
print(f"{'='*80}")

EXTRACTING TRIAL SIZE AND REPEAT DATA

📊 Extracting data for ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ...
  ✓ Extracted 1,221,083 records

📊 Extracting data for ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ...
  ✓ Extracted 1,285,379 records

📊 Extracting data for ｱﾘｴｰﾙｼﾞｪﾙ...
  ✓ Extracted 1,032,856 records

📊 Extracting data for ﾎﾞｰﾙﾄﾞｼﾞｪﾙ...
  ✓ Extracted 322,784 records

📊 Extracting data for ｱﾀｯｸ抗菌EX...
  ✓ Extracted 1,596,304 records

📊 Extracting data for ﾅﾉｯｸｽﾜﾝ...
  ✓ Extracted 373,364 records

✓ Data extraction completed in 2645.3 seconds
  Total records: 5,831,770
  Unique shoppers: 4,541,101
  Initiatives: 12


In [8]:
# =============================================================================
# Phase 1 Analysis: 6-Month Repeat Rate by Individual Size
# =============================================================================

print("="*80)
print("PHASE 1 RESULTS: 6-MONTH REPEAT RATE BY INDIVIDUAL TRIAL SIZE")
print("="*80)

# サイズの順序（大→小）
size_order = [
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ', '詰替超ﾃﾗｼﾞｬﾝﾎﾞ', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', 
    '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', 
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｼﾞｬﾝﾎﾞ', '詰替超特大',
    '本体大', '本体通常', '詰替通常', 'ｿﾉﾀ'
]

# Calculate repeat rate by initiative and size
size_repeat_results = []

for init in initiatives:
    init_name = init['initiative_name']
    init_data = trial_repeat_df[trial_repeat_df['initiative_name'] == init_name]
    
    if len(init_data) == 0:
        continue
    
    # Group by trial size
    size_stats = init_data.groupby('trial_size').agg(
        trial_count=('shopper_key', 'nunique'),
        repeat_count=('has_repeat', 'sum')
    ).reset_index()
    
    size_stats['repeat_rate'] = size_stats['repeat_count'] / size_stats['trial_count'] * 100
    size_stats['initiative'] = init_name
    size_stats['has_6month_data'] = init['has_6month_data']
    size_stats['observation_months'] = init['observation_months']
    
    size_repeat_results.append(size_stats)

# Combine results
size_repeat_df = pd.concat(size_repeat_results, ignore_index=True)

# Create pivot table
pivot_repeat = size_repeat_df.pivot_table(
    index='trial_size',
    columns='initiative',
    values='repeat_rate',
    aggfunc='first'
)

# Reorder by size
existing_sizes = [s for s in size_order if s in pivot_repeat.index]
other_sizes = [s for s in pivot_repeat.index if s not in size_order]
pivot_repeat = pivot_repeat.reindex(existing_sizes + other_sizes)

# Add observation status to column names
init_status = {i['initiative_name']: '✅' if i['has_6month_data'] else '⚠️' for i in initiatives}
new_columns = [f"{init_status.get(col, '')} {col}" for col in pivot_repeat.columns]
pivot_repeat.columns = new_columns

print("\n📊 6-Month Repeat Rate (%) by Individual Trial Size")
print("   ✅ = Full 6-month data | ⚠️ = Observation period < 6 months")
print("-"*80)
display(pivot_repeat.round(1))

# Also show trial counts
pivot_count = size_repeat_df.pivot_table(
    index='trial_size',
    columns='initiative',
    values='trial_count',
    aggfunc='first'
).reindex(existing_sizes + other_sizes)

print("\n📊 Trial Count by Individual Size")
print("-"*80)
display(pivot_count)

PHASE 1 RESULTS: 6-MONTH REPEAT RATE BY INDIVIDUAL TRIAL SIZE

📊 6-Month Repeat Rate (%) by Individual Trial Size
   ✅ = Full 6-month data | ⚠️ = Observation period < 6 months
--------------------------------------------------------------------------------


,⚠️ Anakin (All),✅ Ariel Gel (Liquid),⚠️ Attack Antibacterial EX,⚠️ Bold Gel (Liquid),✅ Cinderella (All),✅ Moana,⚠️ Nanox One,⚠️ Rapunzel (All),✅ Snowwhite,✅ Srixon,✅ Srixon Boost,✅ Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,27.9,NaN,NaN,NaN,72.3,73.2,NaN,42.4,62.2,75.1,74.4,65.3
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,27.8,NaN,NaN,NaN,NaN,NaN,NaN,47.3,66.4,NaN,NaN,67.5
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,45.2,NaN,NaN,NaN,73.5,74.0,NaN,53.4,67.2,75.4,75.2,65.0
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,40.7,NaN,NaN,NaN,69.7,68.5,NaN,50.1,62.8,71.1,74.3,61.2
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,39.7,64.5,65.1,57.3,70.3,73.5,41.0,50.0,64.3,74.6,73.2,62.3
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,71.6,70.5,NaN,50.0,60.8,NaN,NaN,NaN,0.0,100.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,67.3,71.4,56.5,NaN,100.0,53.4,NaN,NaN,100.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,14.3,49.6,NaN,61.1,68.5,68.6,53.0,75.0,37.5,66.9,58.8,56.7
詰替超特大,0.0,64.2,70.4,52.8,61.5,51.4,57.1,NaN,0.0,52.0,66.7,66.7



📊 Trial Count by Individual Size
--------------------------------------------------------------------------------


initiative,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,30670.0,NaN,NaN,NaN,16587.0,19500.0,NaN,24454.0,27937.0,18474.0,16088.0,18669.0
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,24285.0,NaN,NaN,NaN,NaN,NaN,NaN,16031.0,4476.0,NaN,NaN,3688.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,98862.0,NaN,NaN,NaN,110996.0,125862.0,NaN,120190.0,111264.0,123585.0,122915.0,123130.0
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,28122.0,NaN,NaN,NaN,41689.0,51996.0,NaN,26339.0,35537.0,48332.0,45488.0,46357.0
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,37991.0,81258.0,145499.0,25360.0,38681.0,27415.0,21194.0,34318.0,41994.0,44672.0,42731.0,48365.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,173646.0,615205.0,NaN,2.0,1107.0,NaN,NaN,NaN,2.0,1.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,219438.0,330379.0,62879.0,NaN,1.0,129453.0,NaN,NaN,2.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,14.0,1609.0,NaN,18481.0,184.0,12797.0,87512.0,4.0,32.0,7212.0,503.0,150.0
詰替超特大,2.0,262061.0,369194.0,95124.0,13.0,208.0,33439.0,NaN,2.0,25.0,3.0,3.0


## 3. Phase 2: Time-to-Repeat Analysis

### 目的
- サイズ別「リピートまでの平均日数」を算出
- 大きいサイズの消費期間が長いことを定量化
- 比較可能な期間（最初の2ヶ月）での評価

In [9]:
# =============================================================================
# Phase 2: Time-to-Repeat Analysis
# サイズ別リピートまでの日数を分析
# =============================================================================

print("="*80)
print("PHASE 2: TIME-TO-REPEAT ANALYSIS")
print("サイズ別「リピートまでの平均日数」")
print("="*80)

# Calculate days to first repeat
trial_repeat_df['first_trial_date'] = pd.to_datetime(trial_repeat_df['first_trial_date'])
trial_repeat_df['first_repeat_date'] = pd.to_datetime(trial_repeat_df['first_repeat_date'])

# Only for those who repeated
repeaters = trial_repeat_df[trial_repeat_df['has_repeat'] == 1].copy()
repeaters['days_to_repeat'] = (repeaters['first_repeat_date'] - repeaters['first_trial_date']).dt.days

# Aggregate by size
time_to_repeat_by_size = repeaters.groupby('trial_size').agg(
    repeater_count=('shopper_key', 'nunique'),
    mean_days=('days_to_repeat', 'mean'),
    median_days=('days_to_repeat', 'median'),
    min_days=('days_to_repeat', 'min'),
    max_days=('days_to_repeat', 'max'),
    std_days=('days_to_repeat', 'std')
).reset_index()

# Reorder by size
time_to_repeat_by_size['size_order'] = time_to_repeat_by_size['trial_size'].apply(
    lambda x: size_order.index(x) if x in size_order else 999
)
time_to_repeat_by_size = time_to_repeat_by_size.sort_values('size_order').drop('size_order', axis=1)

print("\n📊 Days to First Repeat by Trial Size")
print("-"*80)
print(f"{'Trial Size':<20} | {'N':<8} | {'Mean':<8} | {'Median':<8} | {'Min':<6} | {'Max':<6}")
print("-"*80)

for _, row in time_to_repeat_by_size.iterrows():
    print(f"{row['trial_size']:<20} | {int(row['repeater_count']):<8,} | {row['mean_days']:<8.1f} | {row['median_days']:<8.1f} | {int(row['min_days']):<6} | {int(row['max_days']):<6}")

# Visualize
fig = px.bar(
    time_to_repeat_by_size,
    x='trial_size',
    y='median_days',
    title='📊 Median Days to First Repeat by Trial Size<br><sub>大きいサイズほど消費期間が長いためリピートまでの日数が長い</sub>',
    labels={'trial_size': 'Trial Size', 'median_days': 'Median Days to Repeat'},
    text='median_days'
)
fig.update_traces(texttemplate='%{text:.0f}日', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

PHASE 2: TIME-TO-REPEAT ANALYSIS
サイズ別「リピートまでの平均日数」

📊 Days to First Repeat by Trial Size
--------------------------------------------------------------------------------
Trial Size           | N        | Mean     | Median   | Min    | Max   
--------------------------------------------------------------------------------
詰替ﾃﾗｼﾞｬﾝﾎﾞ           | 86,061   | 104.9    | 72.0     | 1      | 721   
詰替超ﾃﾗｼﾞｬﾝﾎﾞ          | 18,523   | 63.1     | 53.0     | 1      | 330   
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ        | 458,182  | 79.1     | 48.0     | 1      | 725   
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ         | 167,706  | 102.8    | 67.0     | 1      | 722   
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          | 339,033  | 81.6     | 60.0     | 1      | 726   
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ        | 555,407  | 62.3     | 49.0     | 1      | 722   
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         | 484,695  | 58.6     | 46.0     | 1      | 364   
詰替超ｼﾞｬﾝﾎﾞ            | 72,349   | 63.0     | 45.0     | 1      | 716   
詰替超特大                | 488,990  | 56.2     | 42.0     | 1      | 691   
本体大                  | 39,825

In [10]:
# =============================================================================
# Phase 2b: Comparable Period Analysis
# 比較可能期間（最初の2ヶ月）での評価
# =============================================================================

print("="*80)
print("PHASE 2b: COMPARABLE PERIOD ANALYSIS")
print("全商品を「最初の2ヶ月」で比較")
print("="*80)

# Calculate repeat rate at different time windows
def calculate_repeat_rate_at_window(df, window_days):
    """Calculate repeat rate within specified window"""
    df = df.copy()
    df['repeat_within_window'] = (
        (df['has_repeat'] == 1) & 
        (df['days_to_repeat'] <= window_days)
    ).astype(int) if 'days_to_repeat' in df.columns else 0
    
    return df.groupby(['initiative', 'trial_size']).agg(
        trial_count=('shopper_key', 'nunique'),
        repeat_count=('repeat_within_window', 'sum')
    ).reset_index()

# Add days_to_repeat to main df (for those who haven't repeated, set to NaN)
trial_repeat_df_with_days = trial_repeat_df.copy()
trial_repeat_df_with_days['days_to_repeat'] = np.where(
    trial_repeat_df_with_days['has_repeat'] == 1,
    (trial_repeat_df_with_days['first_repeat_date'] - trial_repeat_df_with_days['first_trial_date']).dt.days,
    np.nan
)

# Rename column for consistency
trial_repeat_df_with_days = trial_repeat_df_with_days.rename(columns={'initiative_name': 'initiative'})

# Calculate at 60 days (2 months) and 180 days (6 months)
results_60d = calculate_repeat_rate_at_window(trial_repeat_df_with_days, 60)
results_60d['repeat_rate_60d'] = results_60d['repeat_count'] / results_60d['trial_count'] * 100
results_60d = results_60d.rename(columns={'repeat_count': 'repeat_count_60d'})

results_180d = calculate_repeat_rate_at_window(trial_repeat_df_with_days, 180)
results_180d['repeat_rate_180d'] = results_180d['repeat_count'] / results_180d['trial_count'] * 100
results_180d = results_180d.rename(columns={'repeat_count': 'repeat_count_180d', 'trial_count': 'trial_count_check'})

# Merge
comparison_df = results_60d.merge(
    results_180d[['initiative', 'trial_size', 'repeat_rate_180d']], 
    on=['initiative', 'trial_size'], 
    how='left'
)
comparison_df['rate_increase'] = comparison_df['repeat_rate_180d'] - comparison_df['repeat_rate_60d']

# Add observation status
init_status_dict = {i['initiative_name']: i['has_6month_data'] for i in initiatives}
comparison_df['has_6month_data'] = comparison_df['initiative'].map(init_status_dict)

print("\n📊 Repeat Rate Comparison: 2-Month vs 6-Month Window")
print("   Shows how repeat rate grows over time for each size")
print("-"*80)

# Focus on key sizes and initiatives with full data
key_sizes = ['本体通常', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ']
full_data_initiatives = [i['initiative_name'] for i in initiatives if i['has_6month_data']]

comparison_display = comparison_df[
    (comparison_df['trial_size'].isin(key_sizes)) &
    (comparison_df['has_6month_data'] == True)
].copy()

if len(comparison_display) > 0:
    # Pivot for display
    pivot_60d = comparison_display.pivot_table(index='trial_size', columns='initiative', values='repeat_rate_60d')
    pivot_180d = comparison_display.pivot_table(index='trial_size', columns='initiative', values='repeat_rate_180d')
    pivot_increase = comparison_display.pivot_table(index='trial_size', columns='initiative', values='rate_increase')
    
    print("\n【2ヶ月時点のリピート率 (%)】")
    display(pivot_60d.round(1))
    
    print("\n【6ヶ月時点のリピート率 (%)】")
    display(pivot_180d.round(1))
    
    print("\n【成長幅 (6ヶ月 - 2ヶ月) (pt)】")
    display(pivot_increase.round(1))
else:
    print("⚠ Insufficient data for comparison")

PHASE 2b: COMPARABLE PERIOD ANALYSIS
全商品を「最初の2ヶ月」で比較

📊 Repeat Rate Comparison: 2-Month vs 6-Month Window
   Shows how repeat rate grows over time for each size
--------------------------------------------------------------------------------

【2ヶ月時点のリピート率 (%)】


initiative,Ariel Gel (Liquid),Cinderella (All),Moana,Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,
本体通常,33.6,31.9,28.6,33.0,28.6,35.8,32.0
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,39.9,NaN,0.0,NaN,0.0,NaN,NaN
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,NaN,39.1,38.2,39.7,39.7,44.2,40.6
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,27.4,30.8,31.6,32.4,31.5,34.4,33.6



【6ヶ月時点のリピート率 (%)】


initiative,Ariel Gel (Liquid),Cinderella (All),Moana,Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,
本体通常,51.0,49.3,43.3,48.0,45.1,52.1,40.5
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,66.0,NaN,0.0,NaN,100.0,NaN,NaN
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,NaN,62.7,61.8,62.3,64.5,67.1,61.4
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,63.2,57.5,59.4,58.7,61.1,62.9,58.0



【成長幅 (6ヶ月 - 2ヶ月) (pt)】


initiative,Ariel Gel (Liquid),Cinderella (All),Moana,Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,
本体通常,17.3,17.3,14.7,15.1,16.5,16.3,8.5
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,26.1,NaN,0.0,NaN,100.0,NaN,NaN
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,NaN,23.5,23.5,22.6,24.7,22.9,20.7
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,35.8,26.7,27.7,26.3,29.6,28.5,24.4


## 4. Phase 3: Retention (定着率) Analysis

### 定着率の定義
- **2回以上リピート**: リピート期間内に2回以上購入した人の割合
- 単なる「1回リピート」ではなく、継続的な購入を測定

In [11]:
# =============================================================================
# Phase 3: Retention Analysis
# 定着率 = リピート2回以上の人の割合
# =============================================================================

print("="*80)
print("PHASE 3: RETENTION ANALYSIS")
print("定着率 = リピート期間内に2回以上購入した人の割合")
print("="*80)

# Calculate retention (2+ repeats)
retention_df = trial_repeat_df.copy()
retention_df['is_retained'] = (retention_df['repeat_count'] >= 2).astype(int)

# Aggregate by initiative and size
retention_by_size = retention_df.groupby(['initiative_name', 'trial_size']).agg(
    trial_count=('shopper_key', 'nunique'),
    repeat_1plus=('has_repeat', 'sum'),
    repeat_2plus=('is_retained', 'sum')
).reset_index()

retention_by_size['repeat_rate'] = retention_by_size['repeat_1plus'] / retention_by_size['trial_count'] * 100
retention_by_size['retention_rate'] = retention_by_size['repeat_2plus'] / retention_by_size['trial_count'] * 100
retention_by_size['retention_ratio'] = retention_by_size['repeat_2plus'] / retention_by_size['repeat_1plus'] * 100

# Add observation status
retention_by_size['has_6month_data'] = retention_by_size['initiative_name'].map(init_status_dict)

print("\n📊 Retention Rate (2+ Purchases) by Trial Size")
print("   リピート1回のみ vs 2回以上（定着）の比較")
print("-"*80)

# Pivot tables
pivot_repeat_rate = retention_by_size.pivot_table(
    index='trial_size', columns='initiative_name', values='repeat_rate'
)
pivot_retention_rate = retention_by_size.pivot_table(
    index='trial_size', columns='initiative_name', values='retention_rate'
)
pivot_retention_ratio = retention_by_size.pivot_table(
    index='trial_size', columns='initiative_name', values='retention_ratio'
)

# Reorder
existing_sizes = [s for s in size_order if s in pivot_repeat_rate.index]
pivot_repeat_rate = pivot_repeat_rate.reindex(existing_sizes)
pivot_retention_rate = pivot_retention_rate.reindex(existing_sizes)
pivot_retention_ratio = pivot_retention_ratio.reindex(existing_sizes)

print("\n【リピート率 (1回以上) %】")
display(pivot_repeat_rate.round(1))

print("\n【定着率 (2回以上) %】")
display(pivot_retention_rate.round(1))

print("\n【定着比率 (定着者/リピーター) %】")
print("   リピートした人のうち、2回以上購入した人の割合")
display(pivot_retention_ratio.round(1))

PHASE 3: RETENTION ANALYSIS
定着率 = リピート期間内に2回以上購入した人の割合

📊 Retention Rate (2+ Purchases) by Trial Size
   リピート1回のみ vs 2回以上（定着）の比較
--------------------------------------------------------------------------------

【リピート率 (1回以上) %】


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,27.9,NaN,NaN,NaN,72.3,73.2,NaN,42.4,62.2,75.1,74.4,65.3
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,27.8,NaN,NaN,NaN,NaN,NaN,NaN,47.3,66.4,NaN,NaN,67.5
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,45.2,NaN,NaN,NaN,73.5,74.0,NaN,53.4,67.2,75.4,75.2,65.0
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,40.7,NaN,NaN,NaN,69.7,68.5,NaN,50.1,62.8,71.1,74.3,61.2
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,39.7,64.5,65.1,57.3,70.3,73.5,41.0,50.0,64.3,74.6,73.2,62.3
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,71.6,70.5,NaN,50.0,60.8,NaN,NaN,NaN,0.0,100.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,67.3,71.4,56.5,NaN,100.0,53.4,NaN,NaN,100.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,14.3,49.6,NaN,61.1,68.5,68.6,53.0,75.0,37.5,66.9,58.8,56.7
詰替超特大,0.0,64.2,70.4,52.8,61.5,51.4,57.1,NaN,0.0,52.0,66.7,66.7



【定着率 (2回以上) %】


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,8.0,NaN,NaN,NaN,54.1,56.7,NaN,17.9,41.2,59.5,55.6,44.0
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,5.6,NaN,NaN,NaN,NaN,NaN,NaN,17.0,45.4,NaN,NaN,45.3
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,18.9,NaN,NaN,NaN,58.5,60.0,NaN,29.0,49.8,62.1,60.5,47.3
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,11.9,NaN,NaN,NaN,53.2,53.0,NaN,22.7,43.6,56.6,58.4,42.1
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,13.5,42.4,40.7,29.2,54.5,58.8,13.0,24.4,45.7,60.6,57.3,43.8
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,55.7,51.0,NaN,0.0,44.4,NaN,NaN,NaN,0.0,100.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,50.9,52.3,33.6,NaN,100.0,25.0,NaN,NaN,100.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,0.0,31.7,NaN,40.8,53.3,54.0,26.1,50.0,21.9,52.4,40.6,36.0
詰替超特大,0.0,47.6,51.7,32.3,23.1,33.2,31.1,NaN,0.0,36.0,33.3,33.3



【定着比率 (定着者/リピーター) %】
   リピートした人のうち、2回以上購入した人の割合


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,28.8,NaN,NaN,NaN,74.9,77.5,NaN,42.2,66.2,79.3,74.8,67.4
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,20.2,NaN,NaN,NaN,NaN,NaN,NaN,36.0,68.3,NaN,NaN,67.1
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,41.7,NaN,NaN,NaN,79.6,81.0,NaN,54.4,74.1,82.3,80.5,72.8
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,29.3,NaN,NaN,NaN,76.3,77.4,NaN,45.4,69.5,79.6,78.6,68.9
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,34.0,65.8,62.5,50.9,77.4,80.0,31.7,48.9,71.0,81.2,78.3,70.3
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,77.7,72.3,NaN,0.0,73.0,NaN,NaN,NaN,NaN,100.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,75.6,73.3,59.5,NaN,100.0,46.7,NaN,NaN,100.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,0.0,63.9,NaN,66.9,77.8,78.8,49.3,66.7,58.3,78.3,68.9,63.5
詰替超特大,NaN,74.1,73.5,61.2,37.5,64.5,54.4,NaN,NaN,69.2,50.0,50.0


## 5. Phase 4: Share of Wallet (SOW) Analysis

### 目的
- トライアル製品への支出額 / カテゴリ全体への支出額
- サイズ別のSOWを比較

In [20]:
# =============================================================================
# Phase 4: Share of Wallet Analysis
# Note: This requires additional query for category-level spending
# =============================================================================

print("="*80)
print("PHASE 4: SHARE OF WALLET ANALYSIS")
print("カテゴリ内での当該製品への支出割合")
print("="*80)

# SOW calculation requires separate query for total category spend
# This is a simplified version using repeat_spend from existing data

# For now, we'll calculate relative spending intensity
spend_analysis = trial_repeat_df.groupby(['initiative_name', 'trial_size']).agg(
    trial_count=('shopper_key', 'nunique'),
    repeaters=('has_repeat', 'sum'),
    total_repeat_spend=('repeat_spend', 'sum'),
    avg_repeat_spend=('repeat_spend', 'mean')
).reset_index()

spend_analysis['repeat_rate'] = spend_analysis['repeaters'] / spend_analysis['trial_count'] * 100
# Convert to float to avoid Decimal/float type mismatch
spend_analysis['total_repeat_spend'] = pd.to_numeric(spend_analysis['total_repeat_spend'], errors='coerce')
spend_analysis['repeaters'] = pd.to_numeric(spend_analysis['repeaters'], errors='coerce')
# Calculate with mask for zero repeaters
mask = spend_analysis['repeaters'] > 0
spend_analysis['spend_per_repeater'] = np.nan
spend_analysis.loc[mask, 'spend_per_repeater'] = (
    spend_analysis.loc[mask, 'total_repeat_spend'] / spend_analysis.loc[mask, 'repeaters']
)

# Add observation status
spend_analysis['has_6month_data'] = spend_analysis['initiative_name'].map(init_status_dict)

print("\n📊 Average Repeat Spend per Repeater by Trial Size")
print("   リピーター1人あたりの平均リピート支出額")
print("-"*80)

pivot_spend = spend_analysis.pivot_table(
    index='trial_size', columns='initiative_name', values='spend_per_repeater'
)
existing_sizes = [s for s in size_order if s in pivot_spend.index]
pivot_spend = pivot_spend.reindex(existing_sizes)

display(pivot_spend.round(0))

print("\n💡 Insight: 大きいサイズでトライアルした人は、リピート時の支出額も高い傾向")

PHASE 4: SHARE OF WALLET ANALYSIS
カテゴリ内での当該製品への支出割合

📊 Average Repeat Spend per Repeater by Trial Size
   リピーター1人あたりの平均リピート支出額
--------------------------------------------------------------------------------


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,12236.0,NaN,NaN,NaN,14579.0,17352.0,NaN,20607.0,30828.0,18732.0,14130.0,37983.0
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,4252.0,NaN,NaN,NaN,NaN,NaN,NaN,5367.0,11638.0,NaN,NaN,9614.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,3591.0,NaN,NaN,NaN,14167.0,17364.0,NaN,5232.0,10141.0,16946.0,13617.0,9672.0
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,9574.0,NaN,NaN,NaN,30864.0,34662.0,NaN,16499.0,24511.0,35554.0,29178.0,20826.0
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,7857.0,5408.0,4540.0,3653.0,31404.0,53372.0,3037.0,13635.0,21247.0,35650.0,29331.0,19630.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,6073.0,4054.0,NaN,1078.0,9152.0,NaN,NaN,NaN,NaN,6189.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,5429.0,4641.0,3153.0,NaN,5179.0,3129.0,NaN,NaN,5489.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,882.0,3146.0,NaN,4481.0,4557174.0,104668.0,3442.0,3821.0,3247.0,176737.0,2012345.0,4123667.0
詰替超特大,NaN,3986.0,3927.0,2067.0,1006.0,7522914.0,5592.0,NaN,NaN,3656.0,2442.0,1406.0



💡 Insight: 大きいサイズでトライアルした人は、リピート時の支出額も高い傾向


## 6. Phase 5: Customer Lifetime Value (CLV) Estimation

### 計算式
CLV = 平均購入回数 × 平均購入単価（観測期間内）

In [21]:
# =============================================================================
# Phase 5: CLV Estimation
# =============================================================================

print("="*80)
print("PHASE 5: CUSTOMER LIFETIME VALUE (CLV) ESTIMATION")
print("観測期間内の顧客価値を推定")
print("="*80)

# Calculate CLV metrics
clv_analysis = trial_repeat_df.groupby(['initiative_name', 'trial_size']).agg(
    trial_count=('shopper_key', 'nunique'),
    total_repeat_count=('repeat_count', 'sum'),
    total_repeat_spend=('repeat_spend', 'sum')
).reset_index()

clv_analysis['avg_purchases_per_trialer'] = clv_analysis['total_repeat_count'] / clv_analysis['trial_count']
clv_analysis['avg_spend_per_trialer'] = clv_analysis['total_repeat_spend'] / clv_analysis['trial_count']
# Convert to float to avoid Decimal/float type mismatch
clv_analysis['total_repeat_spend'] = pd.to_numeric(clv_analysis['total_repeat_spend'], errors='coerce')
clv_analysis['total_repeat_count'] = pd.to_numeric(clv_analysis['total_repeat_count'], errors='coerce')
# Calculate with mask for zero repeat count
mask = clv_analysis['total_repeat_count'] > 0
clv_analysis['avg_spend_per_purchase'] = np.nan
clv_analysis.loc[mask, 'avg_spend_per_purchase'] = (
    clv_analysis.loc[mask, 'total_repeat_spend'] / clv_analysis.loc[mask, 'total_repeat_count']
)

# Add observation status  
clv_analysis['has_6month_data'] = clv_analysis['initiative_name'].map(init_status_dict)

print("\n📊 Average Repeat Spend per Trial Shopper (CLV Proxy)")
print("   トライアル者1人あたりの平均リピート売上")
print("-"*80)

pivot_clv = clv_analysis.pivot_table(
    index='trial_size', columns='initiative_name', values='avg_spend_per_trialer'
)
existing_sizes = [s for s in size_order if s in pivot_clv.index]
pivot_clv = pivot_clv.reindex(existing_sizes)

display(pivot_clv.round(0))

print("\n📊 Average Purchases per Trial Shopper")
print("   トライアル者1人あたりの平均リピート回数")
print("-"*80)

pivot_purchases = clv_analysis.pivot_table(
    index='trial_size', columns='initiative_name', values='avg_purchases_per_trialer'
)
pivot_purchases = pivot_purchases.reindex(existing_sizes)

display(pivot_purchases.round(2))

PHASE 5: CUSTOMER LIFETIME VALUE (CLV) ESTIMATION
観測期間内の顧客価値を推定

📊 Average Repeat Spend per Trial Shopper (CLV Proxy)
   トライアル者1人あたりの平均リピート売上
--------------------------------------------------------------------------------


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,3412.279426,NaN,NaN,NaN,10533.862121,12705.134051,NaN,8743.497751,19181.998532,14062.963733,10507.733217,24785.017409
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,1181.591476,NaN,NaN,NaN,NaN,NaN,NaN,2537.47352,7727.517203,NaN,NaN,6488.277928
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,1623.283567,NaN,NaN,NaN,10415.243612,12846.677178,NaN,2793.280215,6815.229023,12785.843541,10233.031282,6285.190409
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,3896.581004,NaN,NaN,NaN,21507.939265,23731.629798,NaN,8262.725692,15385.115598,25295.374679,21676.271654,12738.588735
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,3117.575794,3489.058197,2953.861951,2094.945702,22092.767043,39222.204815,1244.841276,6814.372866,13664.218007,26584.848317,21481.56624,12225.971426
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,4348.308029,2859.015253,NaN,539.0,5564.182475,NaN,NaN,NaN,0.0,6189.0,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,3652.572107,3311.382542,1781.341752,NaN,5179.0,1672.082501,NaN,NaN,5489.0,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,126.0,1560.256681,NaN,2735.789838,3120673.211957,71754.975229,1825.032007,2865.75,1217.75,118289.967138,1184203.101392,2336744.806667
詰替超特大,0.0,2559.015657,2763.694665,1091.095065,619.076923,3869960.548077,3195.417058,NaN,0.0,1901.04,1627.666667,937.333333



📊 Average Purchases per Trial Shopper
   トライアル者1人あたりの平均リピート回数
--------------------------------------------------------------------------------


initiative_name,Anakin (All),Ariel Gel (Liquid),Attack Antibacterial EX,Bold Gel (Liquid),Cinderella (All),Moana,Nanox One,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
trial_size,,,,,,,,,,,,
詰替ﾃﾗｼﾞｬﾝﾎﾞ,0.39,NaN,NaN,NaN,2.82,3.46,NaN,0.76,1.91,3.44,2.66,1.91
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,0.35,NaN,NaN,NaN,NaN,NaN,NaN,0.74,1.96,NaN,NaN,1.84
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,0.75,NaN,NaN,NaN,4.27,5.12,NaN,1.17,2.83,5.28,4.31,2.61
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,0.57,NaN,NaN,NaN,3.30,3.84,NaN,0.93,2.15,3.99,3.48,1.97
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,0.59,1.78,1.66,1.10,3.63,4.61,0.60,0.99,2.41,4.60,3.63,2.18
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,3.08,2.39,NaN,0.50,2.95,NaN,NaN,NaN,0.00,2.00,NaN
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,NaN,2.84,2.60,1.37,NaN,3.00,0.96,NaN,NaN,3.00,NaN,NaN
詰替超ｼﾞｬﾝﾎﾞ,0.14,1.61,NaN,1.74,5.97,4.83,1.01,2.25,1.53,4.71,3.48,3.25
詰替超特大,0.00,2.79,2.70,1.37,1.00,5.33,1.18,NaN,0.00,1.76,1.00,1.33


## 7. Phase 6: Size Migration Analysis

### 目的
- トライアル時サイズ → リピート時サイズの遷移パターン
- アップグレード率・ダウングレード率の算出

In [16]:
# =============================================================================
# Phase 6: Size Migration Analysis
# Note: This requires repeat size data which needs additional query
# =============================================================================

print("="*80)
print("PHASE 6: SIZE MIGRATION ANALYSIS")
print("トライアル時サイズ → リピート時サイズの遷移")
print("="*80)

print("\n⚠ This analysis requires additional query to get repeat purchase size")
print("   Will be implemented in the next iteration with separate data extraction")

# Placeholder for size migration analysis
# This would require:
# 1. Query to get repeat purchase sizes for each shopper
# 2. Compare trial_size vs repeat_size
# 3. Calculate upgrade/downgrade/same rates

PHASE 6: SIZE MIGRATION ANALYSIS
トライアル時サイズ → リピート時サイズの遷移

⚠ This analysis requires additional query to get repeat purchase size
   Will be implemented in the next iteration with separate data extraction


## 8. Phase 7: Cohort Analysis (Monthly Tracking)

### 目的
- トライアル後の月次リピート率推移
- 大サイズが時間とともに追いつく様子を可視化

In [17]:
# =============================================================================
# Phase 7: Cohort Analysis - Monthly Repeat Rate Tracking
# =============================================================================

print("="*80)
print("PHASE 7: COHORT ANALYSIS - MONTHLY REPEAT RATE TRACKING")
print("トライアル後の月次累積リピート率推移")
print("="*80)

# Calculate cumulative repeat rate at each month
def calculate_monthly_cohort(df):
    """Calculate cumulative repeat rate at 1, 2, 3, 4, 5, 6 months"""
    results = []
    
    for month in [1, 2, 3, 4, 5, 6]:
        days = month * 30
        
        # Filter to initiatives with enough observation
        df_filtered = df.copy()
        
        # Calculate repeat within this window
        df_filtered['repeat_in_window'] = (
            (df_filtered['has_repeat'] == 1) & 
            (df_filtered['days_to_repeat'] <= days)
        ).fillna(False).astype(int)
        
        monthly_stats = df_filtered.groupby(['initiative_name', 'trial_size']).agg(
            trial_count=('shopper_key', 'nunique'),
            repeat_count=('repeat_in_window', 'sum')
        ).reset_index()
        
        monthly_stats['repeat_rate'] = monthly_stats['repeat_count'] / monthly_stats['trial_count'] * 100
        monthly_stats['month'] = month
        
        results.append(monthly_stats)
    
    return pd.concat(results, ignore_index=True)

# Prepare data
trial_repeat_with_days = trial_repeat_df.copy()
trial_repeat_with_days['days_to_repeat'] = np.where(
    trial_repeat_with_days['has_repeat'] == 1,
    (trial_repeat_with_days['first_repeat_date'] - trial_repeat_with_days['first_trial_date']).dt.days,
    np.nan
)

# Calculate cohort
cohort_df = calculate_monthly_cohort(trial_repeat_with_days)

# Add observation status
cohort_df['has_6month_data'] = cohort_df['initiative_name'].map(init_status_dict)
cohort_df['obs_months'] = cohort_df['initiative_name'].map(
    {i['initiative_name']: i['observation_months'] for i in initiatives}
)

# Filter to only show months where we have data
cohort_df = cohort_df[cohort_df['month'] <= cohort_df['obs_months']]

print("\n📊 Monthly Cumulative Repeat Rate by Size")
print("   月次累積リピート率の推移（サイズ別）")
print("-"*80)

# Focus on key sizes for visualization
key_sizes = ['本体通常', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ']
full_data_inits = [i['initiative_name'] for i in initiatives if i['has_6month_data']]

# Aggregate across initiatives with full data for cleaner visualization
cohort_agg = cohort_df[
    (cohort_df['trial_size'].isin(key_sizes)) &
    (cohort_df['initiative_name'].isin(full_data_inits))
].groupby(['trial_size', 'month']).agg(
    trial_count=('trial_count', 'sum'),
    repeat_count=('repeat_count', 'sum')
).reset_index()

cohort_agg['repeat_rate'] = cohort_agg['repeat_count'] / cohort_agg['trial_count'] * 100

# Pivot for display
cohort_pivot = cohort_agg.pivot_table(
    index='trial_size', 
    columns='month', 
    values='repeat_rate'
)
cohort_pivot.columns = [f'Month {m}' for m in cohort_pivot.columns]

# Reorder
cohort_pivot = cohort_pivot.reindex([s for s in size_order if s in cohort_pivot.index])

print("\n【Cumulative Repeat Rate (%) by Month】")
print("   ✅ Data from initiatives with full 6-month observation")
display(cohort_pivot.round(1))

# Visualize
if len(cohort_agg) > 0:
    fig = px.line(
        cohort_agg,
        x='month',
        y='repeat_rate',
        color='trial_size',
        title='📊 Cumulative Repeat Rate Growth by Trial Size<br><sub>大きいサイズは出足が遅いが、時間とともに追いつく傾向</sub>',
        labels={'month': 'Months Since Trial', 'repeat_rate': 'Cumulative Repeat Rate (%)', 'trial_size': 'Trial Size'},
        markers=True
    )
    fig.update_layout(height=500)
    fig.show()

PHASE 7: COHORT ANALYSIS - MONTHLY REPEAT RATE TRACKING
トライアル後の月次累積リピート率推移

📊 Monthly Cumulative Repeat Rate by Size
   月次累積リピート率の推移（サイズ別）
--------------------------------------------------------------------------------

【Cumulative Repeat Rate (%) by Month】
   ✅ Data from initiatives with full 6-month observation


,Month 1,Month 2,Month 3,Month 4,Month 5,Month 6
trial_size,,,,,,
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,18.7,40.3,50.7,56.7,60.5,63.3
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,11.1,31.2,44.0,51.9,57.0,60.5
詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,15.8,39.9,51.2,58.6,63.0,66.0
本体通常,20.2,32.5,38.0,42.0,44.8,47.8


## 9. Phase 8: Integrated Summary & Recommendations

### 多角的評価の統合

In [22]:
# =============================================================================
# Phase 8: Integrated Summary
# =============================================================================

print("="*80)
print("PHASE 8: INTEGRATED SUMMARY")
print("多角的分析の統合サマリー")
print("="*80)

# Compile key metrics by size
summary_data = []

for size in size_order:
    size_data = trial_repeat_df[trial_repeat_df['trial_size'] == size]
    
    if len(size_data) == 0:
        continue
    
    # Basic metrics
    trial_count = size_data['shopper_key'].nunique()
    repeat_rate = size_data['has_repeat'].mean() * 100
    retention_rate = (size_data['repeat_count'] >= 2).mean() * 100
    
    # Time to repeat (for repeaters only)
    repeaters_data = size_data[size_data['has_repeat'] == 1]
    if len(repeaters_data) > 0:
        repeaters_data = repeaters_data.copy()
        repeaters_data['days_to_repeat'] = (
            repeaters_data['first_repeat_date'] - repeaters_data['first_trial_date']
        ).dt.days
        median_days = repeaters_data['days_to_repeat'].median()
    else:
        median_days = np.nan
    
    # CLV proxy
    avg_spend = size_data['repeat_spend'].mean()
    avg_purchases = size_data['repeat_count'].mean()
    
    summary_data.append({
        'Trial Size': size,
        'Trial Count': trial_count,
        'Repeat Rate (%)': repeat_rate,
        'Retention Rate (%)': retention_rate,
        'Median Days to Repeat': median_days,
        'Avg Repeat Purchases': avg_purchases,
        'Avg Repeat Spend (¥)': avg_spend
    })

summary_df = pd.DataFrame(summary_data)

print("\n📊 COMPREHENSIVE SUMMARY BY TRIAL SIZE")
print("="*80)
display(summary_df.round(1))

# Key insights
print("\n" + "="*80)
print("🎯 KEY INSIGHTS")
print("="*80)

print("""
1. 【Time-to-Repeat】
   - 大きいサイズほどリピートまでの日数が長い（消費期間の違い）
   - これは短期リピート率が低い主因

2. 【定着率】
   - リピート率と定着率（2回以上）の差を確認
   - 大サイズは「リピートしたら継続する」傾向

3. 【CLV】
   - トライアル者1人あたりの価値を比較
   - 大サイズトライアル者の長期価値

4. 【観測期間の重要性】
   - ⚠️マークの商品は6ヶ月未満のデータ
   - 大サイズの真の価値は長期観測で明らかになる
""")

PHASE 8: INTEGRATED SUMMARY
多角的分析の統合サマリー

📊 COMPREHENSIVE SUMMARY BY TRIAL SIZE


,Trial Size,Trial Count,Repeat Rate (%),Retention Rate (%),Median Days to Repeat,Avg Repeat Purchases,Avg Repeat Spend (¥)
0,詰替ﾃﾗｼﾞｬﾝﾎﾞ,153204,58.4,38.6,72.0,2.0,21556.7
1,詰替超ﾃﾗｼﾞｬﾝﾎﾞ,46261,40.8,16.1,53.0,0.7,6463.0
2,詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,723441,66.6,49.0,48.0,3.4,12242.9
3,詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,269779,64.3,45.7,67.0,2.8,27885.2
4,詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,542068,63.0,41.8,60.0,2.2,17074.7
5,詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,782202,70.7,52.0,49.0,2.5,4509.8
6,詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ,734398,65.8,45.5,46.0,2.3,4556.5
7,詰替超ｼﾞｬﾝﾎﾞ,128018,56.5,32.7,45.0,1.7,48266.2
8,詰替超特大,742366,65.5,46.9,42.0,2.5,5440.0
9,本体大,96023,41.5,18.8,42.0,0.8,3396.6



🎯 KEY INSIGHTS

1. 【Time-to-Repeat】
   - 大きいサイズほどリピートまでの日数が長い（消費期間の違い）
   - これは短期リピート率が低い主因

2. 【定着率】
   - リピート率と定着率（2回以上）の差を確認
   - 大サイズは「リピートしたら継続する」傾向

3. 【CLV】
   - トライアル者1人あたりの価値を比較
   - 大サイズトライアル者の長期価値

4. 【観測期間の重要性】
   - ⚠️マークの商品は6ヶ月未満のデータ
   - 大サイズの真の価値は長期観測で明らかになる



In [23]:
# =============================================================================
# Final Visualization: Multi-Metric Comparison
# =============================================================================

print("="*80)
print("FINAL VISUALIZATION: SIZE LOYALTY SCORECARD")
print("="*80)

# Create a scorecard visualization
if len(summary_df) > 0:
    # Normalize metrics for comparison (0-100 scale)
    summary_norm = summary_df.copy()
    
    # Higher is better for these metrics
    for col in ['Repeat Rate (%)', 'Retention Rate (%)', 'Avg Repeat Purchases', 'Avg Repeat Spend (¥)']:
        if col in summary_norm.columns:
            max_val = summary_norm[col].max()
            if max_val > 0:
                summary_norm[f'{col} (normalized)'] = summary_norm[col] / max_val * 100
    
    # Lower is better for days (invert)
    if 'Median Days to Repeat' in summary_norm.columns:
        max_days = summary_norm['Median Days to Repeat'].max()
        if max_days > 0:
            summary_norm['Repeat Speed (normalized)'] = (1 - summary_norm['Median Days to Repeat'] / max_days) * 100
    
    # Create radar chart data
    metrics_to_plot = [
        'Repeat Rate (%)',
        'Retention Rate (%)',
        'Avg Repeat Purchases',
    ]
    
    # Bar chart comparison
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Repeat Rate (%)', 'Retention Rate (%)', 
                       'Avg Repeat Purchases', 'Median Days to Repeat')
    )
    
    sizes = summary_df['Trial Size'].tolist()[:8]  # Top 8 sizes
    
    fig.add_trace(
        go.Bar(x=sizes, y=summary_df['Repeat Rate (%)'].head(8), name='Repeat Rate'),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=sizes, y=summary_df['Retention Rate (%)'].head(8), name='Retention Rate'),
        row=1, col=2
    )
    fig.add_trace(
        go.Bar(x=sizes, y=summary_df['Avg Repeat Purchases'].head(8), name='Avg Purchases'),
        row=2, col=1
    )
    fig.add_trace(
        go.Bar(x=sizes, y=summary_df['Median Days to Repeat'].head(8), name='Days to Repeat'),
        row=2, col=2
    )
    
    fig.update_layout(
        title='📊 Trial Size Loyalty Scorecard<br><sub>多角的指標でトライアルサイズの価値を評価</sub>',
        height=700,
        showlegend=False
    )
    fig.update_xaxes(tickangle=-45)
    
    fig.show()

print("\n✅ Analysis Complete!")
print("   詳細なプランは COMPREHENSIVE_ANALYSIS_PLAN.md を参照")

FINAL VISUALIZATION: SIZE LOYALTY SCORECARD



✅ Analysis Complete!
   詳細なプランは COMPREHENSIVE_ANALYSIS_PLAN.md を参照
